# Week 11 실습 · 데이터 정제 · 변환 · 집계

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MNU-AI-Programming/AI-Assisted-Python-Programming/blob/main/weeks/week11/practice.ipynb)

**AI활용코딩연습(ZBE135)** · 이영호 교수 · 국립목포대학교 컴퓨터학부

## 오늘의 목표

1. 지저분한 데이터의 문제(중복, 표기 불일치, 단위 섞임, 결측, 이상값)를 **목록으로 진단**합니다.
2. `drop_duplicates`, `.str` 메서드, `replace`/`map`, `to_numeric`, `to_datetime`, `clip`으로 정제합니다.
3. 정제 단계를 **함수로 묶은 파이프라인**을 만들고, 정제 전후 비교표로 검증합니다.
4. `groupby().agg()`, `pivot_table`, `merge`, `pd.cut`으로 집계합니다.
5. AI 정제 코드의 **순서 문제**와 **범주형 결측을 평균으로 채우는 실수**를 찾아냅니다.

| 구간 | 내용 | 시간 |
|---|---|---|
| 1 | 진단: 무엇이 문제인가? | 15분 |
| 2 | Level 1 정제 도구 | 25분 |
| 3 | Level 2 바꾸기 | 10분 |
| 4 | Level 3 정제 파이프라인과 전후 비교 | 25분 |
| 5 | Level 1~3 집계: groupby · pivot · merge | 25분 |
| 6 | Level 4 AI 정제 코드 검증 | 10분 |
| 7 | 정리 | 10분 |

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/MNU-AI-Programming/AI-Assisted-Python-Programming/main/datasets/"

def get_data(name):
    for folder in [Path("../../datasets"), Path("datasets"), Path(".")]:
        if (folder / name).exists():
            return str(folder / name)
    urllib.request.urlretrieve(DATA_URL + name, name)
    return name

raw = pd.read_csv(get_data("survey_raw.csv"), dtype=str, keep_default_na=False)   # 일단 모두 문자열로
print(raw.shape)
raw.head(8)

> 💡 정제 전에는 `dtype=str, keep_default_na=False`로 **원본 그대로** 읽으면 빈 칸이 `""`로 남아 문제를 정확히 볼 수 있습니다.

---
## 1. 진단: 무엇이 문제인가?

In [ ]:
for col in ["department", "grade", "uses_ai", "satisfaction"]:
    print(col, raw[col].value_counts().to_dict(), "\n")

In [ ]:
print("coding_hours 고유값:", sorted(raw["coding_hours"].unique(), key=lambda v: (len(v), v)))
print("완전히 같은 행:", raw.duplicated().sum(), "개 / 응답 ID 중복:", raw["response_id"].duplicated().sum(), "개")
print("날짜 형식 예:", raw["submitted_at"].str[:10].str.replace(r"\d", "0", regex=True).value_counts().to_dict())

### 진단 결과 → 정제 계획

| 열 | 문제 | 정제 방법 |
|---|---|---|
| 전체 | 같은 응답 6건 중복 | `drop_duplicates` |
| `department` | 공백, 띄어쓰기, 줄임말 (`컴공`, `보안학과`) | 공백 제거 → 줄임말 사전으로 통일 |
| `grade` | `"2"`, `"2학년"`, `" 2 "` | 숫자만 추출 → 정수 |
| `coding_hours` | `"2시간"`, `"약 3"`, 빈 칸, `30`, `-1` | 숫자 추출 → 실수, 0~16 밖은 결측 |
| `uses_ai` | 예/Y/yes/사용함, 아니오/N/no/사용안함 | 참/거짓으로 매핑 |
| `satisfaction` | `"4점"`, 빈 칸 | 숫자 추출 → 결측 허용 정수 |
| `submitted_at` | `-`, `/`, `.` 구분자 섞임 | `to_datetime(format="mixed")` |

> **정제 계획을 먼저 표로** 쓰면, AI에게 요청할 때도 그대로 프롬프트가 됩니다.

---
## 2. Level 1 따라하기: 정제 도구

### 2-1. 중복 제거

In [ ]:
df = raw.drop_duplicates().reset_index(drop=True)
print(len(raw), "→", len(df))

### 2-2. `.str` 메서드: 문자열 열 한 번에 다루기

열 전체에 문자열 메서드를 쓰려면 `.str`을 붙입니다. `str.extract(정규식)`은 패턴에 맞는 부분을 꺼냅니다.

| 정규식 | 의미 | 예 |
|---|---|---|
| `\d` | 숫자 한 글자 | `"2학년"` → `2` |
| `\d+` | 숫자 여러 글자 | `"30"` |
| `-?\d+\.?\d*` | 음수·소수 포함 숫자 | `"약 2.5"` → `2.5`, `"-1"` → `-1` |

In [ ]:
print(df["department"].str.strip().str.replace(" ", "").value_counts().to_dict())

grade = df["grade"].str.extract(r"(\d)")[0].astype(int)
print(grade.value_counts().sort_index().to_dict())

hours_text = df["coding_hours"].str.extract(r"(-?\d+\.?\d*)")[0]
print(hours_text.head(10).tolist())

### 2-3. `replace` / `map`: 값 바꾸기

- `replace(사전)`: 사전에 **있는 값만** 바꾸고 나머지는 그대로
- `map(사전)`: 사전에 **없는 값은 NaN** → 빠뜨린 표기를 찾기 좋음

In [ ]:
dept_alias = {"컴공": "컴퓨터공학과", "정보보안": "정보보안학과", "보안학과": "정보보안학과",
              "전자공": "전자공학과", "경영": "경영학과", "간호": "간호학과"}
dept = df["department"].str.strip().str.replace(" ", "").replace(dept_alias)
print(dept.value_counts().to_dict())

ai_map = {"예": True, "Y": True, "yes": True, "사용함": True,
          "아니오": False, "N": False, "no": False, "사용안함": False}
uses_ai = df["uses_ai"].str.strip().map(ai_map)
print("매핑 안 된 값:", uses_ai.isna().sum())

### 2-4. 숫자·날짜 변환과 이상값

- `pd.to_numeric(x, errors="coerce")`: 바꿀 수 없으면 **NaN**
- `where(조건)`: 조건이 거짓인 곳을 NaN으로 (이상값 제거)
- `clip(하한, 상한)`: 범위 밖 값을 경계로 **자르기** (값을 남기고 싶을 때)

In [ ]:
hours = pd.to_numeric(hours_text, errors="coerce")
print("결측:", hours.isna().sum(), "/ 범위 밖:", ((hours < 0) | (hours > 16)).sum())
hours_clean = hours.where(hours.between(0, 16))      # 하루 16시간 넘는 코딩·음수는 응답 오류로 보고 결측 처리
print("정제 후 결측:", hours_clean.isna().sum(), "/ 최대:", hours_clean.max())

sat = pd.to_numeric(df["satisfaction"].str.extract(r"(\d)")[0], errors="coerce").astype("Int64")
print(sat.value_counts(dropna=False).sort_index().to_dict())

when = pd.to_datetime(df["submitted_at"], format="mixed")
print(when.min(), "~", when.max(), "/ 변환 실패:", when.isna().sum())

> `Int64`(대문자 I)는 **결측을 허용하는 정수** 자료형입니다. 그냥 `int`로 바꾸면 NaN 때문에 오류가 나거나 `float`가 됩니다.

---
## 3. Level 2 바꾸기

### ✏️ 직접 해보기

1. `dept_alias`에서 `"보안학과"`를 빼고 `map`으로 바꿔 보세요. 어떤 값이 NaN이 되나요? (`replace`와 `map`의 차이)
2. 코딩 시간의 상한을 16에서 10으로 바꾸면 결측이 몇 개 늘어나나요?
3. `clip(0, 16)`을 쓰면 `where`와 결과가 어떻게 다른지 비교해 보세요.

In [ ]:
alias_test = {k: v for k, v in dept_alias.items() if k != "보안학과"}          # ← 바꿔 보세요
mapped = df["department"].str.strip().str.replace(" ", "").map(
    {**alias_test, **{v: v for v in dept_alias.values()}})
print("map 결과 NaN:", mapped.isna().sum(), df.loc[mapped.isna(), "department"].unique())

print("where:", hours.where(hours.between(0, 16)).describe()[["count", "max"]].to_dict())
print("clip :", hours.clip(0, 16).describe()[["count", "max"]].to_dict())

---
## 4. Level 3 적용하기: 정제 파이프라인과 전후 비교

정제 단계를 **하나의 함수**로 묶으면 새 데이터가 와도 같은 방식으로 정제할 수 있고, 테스트하기도 쉽습니다.

In [ ]:
DEPT_ALIAS = dept_alias
AI_MAP = ai_map


def clean_survey(raw_df):
    """설문 원본(문자열) → 정제된 DataFrame. 원본은 바꾸지 않는다."""
    df = raw_df.drop_duplicates().copy()
    out = pd.DataFrame({
        "response_id": df["response_id"].str.strip(),
        "submitted_at": pd.to_datetime(df["submitted_at"], format="mixed"),
        "department": df["department"].str.strip().str.replace(" ", "").replace(DEPT_ALIAS),
        "grade": df["grade"].str.extract(r"(\d)")[0].astype(int),
        "coding_hours": pd.to_numeric(df["coding_hours"].str.extract(r"(-?\d+\.?\d*)")[0], errors="coerce"),
        "uses_ai": df["uses_ai"].str.strip().map(AI_MAP),
        "satisfaction": pd.to_numeric(df["satisfaction"].str.extract(r"(\d)")[0], errors="coerce").astype("Int64"),
    })
    out["coding_hours"] = out["coding_hours"].where(out["coding_hours"].between(0, 16))
    return out.sort_values("response_id").reset_index(drop=True)


survey = clean_survey(raw)
survey.info()

In [ ]:
survey.head()

### 정제 전후 비교표

In [ ]:
compare = pd.DataFrame({
    "정제 전": [len(raw), raw["department"].nunique(), raw["grade"].nunique(), raw["uses_ai"].nunique(),
              (raw["coding_hours"] == "").sum(), (raw["satisfaction"] == "").sum()],
    "정제 후": [len(survey), survey["department"].nunique(), survey["grade"].nunique(), survey["uses_ai"].nunique(),
              survey["coding_hours"].isna().sum(), survey["satisfaction"].isna().sum()],
}, index=["행 수", "학과 표기 수", "학년 표기 수", "AI 사용 표기 수", "코딩 시간 결측", "만족도 결측"])
compare

코딩 시간 결측이 늘어난 것은 **범위 밖 값(30, 25, −1)을 결측으로 바꿨기 때문**입니다. 이런 결정은 보고서에 반드시 적습니다.

### 정제 결과 검증

In [ ]:
def test_clean_survey():
    s = clean_survey(raw)
    assert len(s) == 150 and s["response_id"].is_unique
    assert set(s["department"]) == {"간호학과", "경영학과", "전자공학과", "정보보안학과", "컴퓨터공학과"}
    assert set(s["grade"]) <= {1, 2, 3, 4}
    assert s["uses_ai"].notna().all()
    assert s["coding_hours"].dropna().between(0, 16).all()
    assert s["satisfaction"].dropna().between(1, 5).all()
    assert raw.shape == (156, 8)                       # 원본은 그대로


test_clean_survey()
print("정제 파이프라인 테스트 통과 ✅")

---
## 5. 집계: groupby · pivot_table · merge · cut

### 5-1. `groupby().agg()`: 그룹별 여러 통계

`새 열 이름=("원래 열", "함수")` 형식으로 쓰면 결과 열 이름이 깔끔합니다.

In [ ]:
by_dept = survey.groupby("department").agg(
    responses=("response_id", "size"),
    avg_hours=("coding_hours", "mean"),
    ai_rate=("uses_ai", "mean"),
    avg_sat=("satisfaction", "mean"),
).round(2)
by_dept.sort_values("avg_hours", ascending=False)

### 5-2. `pivot_table`: 두 기준으로 펼치기

학년(행) × AI 사용 여부(열)별 평균 만족도

In [ ]:
pivot = survey.pivot_table(index="grade", columns="uses_ai", values="satisfaction",
                           aggfunc="mean").round(2)
pivot.columns = ["AI 미사용", "AI 사용"]
pivot

In [ ]:
pd.crosstab(survey["grade"], survey["uses_ai"], margins=True)    # 개수 교차표

### 5-3. `merge`: 다른 표와 합치기

학과가 어느 단과대학에 속하는지는 `departments.csv`에 있습니다. 공통 열 `department`로 합칩니다.

In [ ]:
depts = pd.read_csv(get_data("departments.csv"))
merged = survey.merge(depts, on="department", how="left", validate="many_to_one")
print("합친 뒤 행 수:", len(merged), "/ 단과대 결측:", merged["college"].isna().sum())
merged.groupby("college").agg(responses=("response_id", "size"), avg_hours=("coding_hours", "mean")).round(2)

> `validate="many_to_one"`은 오른쪽 표의 키가 **중복되지 않는지** 확인합니다. 키가 중복되면 행이 몰래 늘어나는 사고를 막아 줍니다.
> 합친 뒤에는 **행 수가 그대로인지** 꼭 확인하세요.

### 5-4. `pd.cut`: 구간으로 나누기

In [ ]:
survey["hours_band"] = pd.cut(survey["coding_hours"], bins=[0, 1, 3, 6, 16],
                              labels=["1시간 이하", "1~3시간", "3~6시간", "6시간 초과"], include_lowest=True)
survey.groupby("hours_band", observed=True).agg(n=("response_id", "size"), sat=("satisfaction", "mean")).round(2)

---
## 6. Level 4 AI와 함께: 정제 코드 검증

AI에게 "설문 데이터를 정제해 줘"라고 요청해 받은 코드에서 흔히 보이는 두 가지 문제입니다.

### 문제 1: 정제 순서 – 표기를 통일하기 전에 중복을 확인

In [ ]:
messy = pd.DataFrame({"name": ["김민준", "김민준 ", "이서연"], "dept": ["컴공", "컴퓨터공학과", "경영학과"]})

ai_way = messy.drop_duplicates()                                   # AI 코드: 먼저 중복 제거
print("AI 방식 행 수:", len(ai_way))

fixed = messy.assign(name=messy["name"].str.strip(),
                     dept=messy["dept"].replace({"컴공": "컴퓨터공학과"})).drop_duplicates()
print("통일 후 중복 제거:", len(fixed))

같은 사람인데 공백·줄임말 때문에 **다른 행으로 보여 중복이 남습니다.** 정제 순서는 보통 **표기 통일 → 중복 제거 → 형 변환 → 이상값 처리**입니다. (이번 설문 데이터는 완전히 같은 행만 중복이라 순서가 결과에 영향을 주지 않았지만, 실제 데이터에서는 자주 문제가 됩니다.)

### 문제 2: 범주형 결측을 평균으로 채우기

In [ ]:
demo = survey[["department", "satisfaction"]].copy()
demo.loc[[0, 1], "department"] = np.nan

try:
    demo["department"].fillna(demo["department"].mean())        # AI 코드
except TypeError as e:
    print("TypeError:", str(e)[:60])

print("최빈값으로 채우기 예:", demo["department"].fillna(demo["department"].mode()[0]).iloc[:2].tolist())
print("또는 '미상'으로 표시:", demo["department"].fillna("미상").iloc[:2].tolist())

만족도처럼 **1~5 순서형 응답**도 평균으로 채우면 `3.47`처럼 존재하지 않는 응답이 생깁니다. 결측 처리는 **열의 성격**에 맞게 정하고, 분석 목적에 따라 **채우지 않고 두는 것**도 좋은 선택입니다.

### 📝 AI에게 정제를 요청할 때

```text
설문 CSV를 정제하는 함수 clean_survey(raw_df)를 작성해 주세요. 원본은 바꾸지 마세요.
정제 계획:
1. department: 공백 제거 후 {"컴공": "컴퓨터공학과", ...}로 통일
2. 표기 통일 후 drop_duplicates
3. coding_hours: 숫자만 추출해 float, 0~16 밖은 NaN (clip 아님)
4. satisfaction: 숫자만 추출해 Int64, 결측은 채우지 않음
마지막에 행 수, 고유값, 범위를 확인하는 assert를 포함하세요.
```

진단 표에서 만든 **정제 계획을 그대로** 넣으면 AI가 임의로 결측을 채우거나 이상값을 자르는 일을 줄일 수 있습니다.

---
## 7. 정리

| 오늘 배운 것 | 핵심 |
|---|---|
| 진단 | `value_counts`, `unique`, `duplicated` → 정제 계획 표 |
| 문자열 정제 | `.str.strip/replace/extract`, 정규식 `\d`, `-?\d+\.?\d*` |
| 값 통일 | `replace`(나머지 유지) vs `map`(없으면 NaN) |
| 형 변환 | `to_numeric(errors="coerce")`, `to_datetime(format="mixed")`, `Int64` |
| 이상값 | `where`(결측으로) vs `clip`(경계로 자르기) |
| 파이프라인 | 정제 함수 + 전후 비교표 + 테스트 |
| 집계 | `groupby().agg(이름=(열, 함수))`, `pivot_table`, `crosstab`, `merge(validate=...)`, `cut` |
| AI 검증 | 정제 순서, 범주형·순서형 결측 처리 |

### 자기 점검

- [ ] 정제 계획 표를 만들 수 있다.
- [ ] `replace`와 `map`의 차이를 설명할 수 있다.
- [ ] `where`와 `clip`의 결과 차이를 말할 수 있다.
- [ ] `merge` 후 행 수를 확인해야 하는 이유를 안다.

### 다음 주 예고

**Week 12 · 탐색적 데이터 분석과 프로젝트 기획**: 캠퍼스 카페 판매 데이터로 EDA를 하고, 최종 프로젝트 연구 질문과 제안서를 만듭니다.